In [1]:
import pandas as pd 

In [2]:
raw_df = pd.read_csv('listings.csv')
raw_df.head(2)

,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,...,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,6422,https://www.airbnb.com/rooms/6422,20250923202825,2025-09-23,city scrape,Nashville Charm,30 day plus rental - book for one month and th...,Historic East Nashville is home to many new an...,https://a0.muscache.com/pictures/miso/Hosting-...,12172,...,4.96,4.92,4.98,NaN,f,1,0,1,0,3.34
1,39870,https://www.airbnb.com/rooms/39870,20250923202825,2025-09-24,city scrape,Close to Vanderbilt 2,"Since I am older, I need for guests to be vacc...","The house is in a safe, quiet, ""college"" neig...",https://a0.muscache.com/pictures/miso/Hosting-...,171184,...,4.97,4.93,4.92,NaN,f,1,0,1,0,5.34


#### Step 1: Analyze

Since the dataset is filled with a ton of columns, we will classify them :
* Core business/ML features (Keep)
* Temporarily (Keep might be useful)
* Almost no analytical value for this project (Drop)

In [23]:
drop_cols = [
    # Identification
    'listing_url',
    'scrape_id',
    'source',
    'host_id',
    'host_url',


    # Images
    'picture_url',
    'host_thumbnail_url',
    'host_picture_url',

    # Long text (NLP project)
    'name',
    'description',
    'neighborhood_overview',
    'host_about',

    # Host information not useful
    'host_name',
    'host_location',
    'host_neighbourhood',
    'host_has_profile_pic',

    # Redundant / Metadata

    'last_scraped',
    'calendar_updated',
    'calendar_last_scraped',
    'has_availability',

    # Redundant bathroom column
    'bathrooms_text',

    # Calendar statistics
    # (too niche for this project)
    'minimum_minimum_nights',
    'maximum_minimum_nights',
    'minimum_maximum_nights',
    'maximum_maximum_nights',
    'minimum_nights_avg_ntm',
    'maximum_nights_avg_ntm',

    # Redundant availability/review columns
    'availability_60',
    'availability_90',
    'availability_eoy',
    'number_of_reviews_ly',

    # Legal
    'license',
    
    # Empty / mostly unused
    'neighbourhood_group_cleansed',
    'calculated_host_listings_count',
    'calculated_host_listings_count_entire_homes',
    'calculated_host_listings_count_private_rooms',
    'calculated_host_listings_count_shared_rooms',
    'number_of_reviews_l30d',
    'first_review',
    'last_review',
    'review_scores_accuracy',
    'review_scores_checkin',
    'review_scores_communication',
    'review_scores_value',
    'neighbourhood',
    'host_listings_count',
    'host_verifications'
]
len(drop_cols)

47

In [28]:
df = raw_df.drop(columns=drop_cols)
df.head(3)

,id,host_since,host_response_time,host_response_rate,host_acceptance_rate,host_is_superhost,host_total_listings_count,host_identity_verified,neighbourhood_cleansed,latitude,longitude,property_type,room_type,accommodates,bathrooms,bedrooms,beds,amenities,price,minimum_nights,maximum_nights,availability_30,availability_365,number_of_reviews,number_of_reviews_ltm,estimated_occupancy_l365d,estimated_revenue_l365d,review_scores_rating,review_scores_cleanliness,review_scores_location,instant_bookable,reviews_per_month
0,6422,2009-04-03,within a day,100%,NaN,f,1.0,t,District 6,36.17143,-86.73570,Private room in home,Private room,1,1.0,1.0,1.0,"[""Dryer \u2013 In building"", ""Kayak"", ""Carbon ...",$43.00,30,365,0,175,667,0,0,0.0,4.95,4.96,4.92,f,3.34
1,39870,2010-07-18,within an hour,100%,91%,t,2.0,t,District 25,36.12466,-86.81269,Private room in home,Private room,2,1.0,1.0,1.0,"[""Carbon monoxide alarm"", ""Microwave"", ""Centra...",$70.00,1,7,17,242,587,80,255,17850.0,4.93,4.91,4.93,f,5.34
2,72906,2010-07-21,within an hour,100%,96%,t,1.0,t,District 18,36.13122,-86.80066,Entire rental unit,Entire home/apt,2,1.0,2.0,2.0,"[""Carbon monoxide alarm"", ""Microwave"", ""First ...",$91.00,2,7,8,234,777,34,204,18564.0,4.92,4.83,4.97,f,4.46


#### Step 2: Data Audit 
A data audit tells you WHAT to clean and HOW to clean it.

In [30]:
df.describe()
df.nunique()
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 9443 entries, 0 to 9442
Data columns (total 32 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   id                         9443 non-null   int64  
 1   host_since                 9432 non-null   str    
 2   host_response_time         8809 non-null   str    
 3   host_response_rate         8809 non-null   str    
 4   host_acceptance_rate       9013 non-null   str    
 5   host_is_superhost          9046 non-null   str    
 6   host_total_listings_count  9432 non-null   float64
 7   host_identity_verified     9432 non-null   str    
 8   neighbourhood_cleansed     9443 non-null   str    
 9   latitude                   9443 non-null   float64
 10  longitude                  9443 non-null   float64
 11  property_type              9443 non-null   str    
 12  room_type                  9443 non-null   str    
 13  accommodates               9443 non-null   int64  
 14  bat

In [31]:
(df.isnull().sum() / len(df) * 100).sort_values(ascending=False)

estimated_revenue_l365d      29.746902
price                        29.746902
bathrooms                    29.662184
beds                         29.662184
review_scores_rating         10.526316
reviews_per_month            10.526316
review_scores_location       10.526316
review_scores_cleanliness    10.526316
host_response_time            6.713968
host_response_rate            6.713968
host_acceptance_rate          4.553638
host_is_superhost             4.204172
bedrooms                      2.403897
host_identity_verified        0.116488
host_total_listings_count     0.116488
host_since                    0.116488
room_type                     0.000000
accommodates                  0.000000
longitude                     0.000000
property_type                 0.000000
latitude                      0.000000
neighbourhood_cleansed        0.000000
id                            0.000000
amenities                     0.000000
number_of_reviews             0.000000
availability_365         

In [7]:
df.duplicated().sum()

np.int64(0)

#### Step 3: Data Cleaning 

* Convert datatypes
* Standardize categorical values(tf to true false)
* Handle missing values
* Remove duplicates
* Handle outliers(Price Bedrooms Bathrooms Revenue)

In [8]:
pd.set_option('display.max_columns', None)
df.head(2)

,id,host_since,host_response_time,host_response_rate,host_acceptance_rate,host_is_superhost,host_total_listings_count,host_verifications,host_identity_verified,neighbourhood_cleansed,latitude,longitude,property_type,room_type,accommodates,bathrooms,bedrooms,beds,amenities,price,minimum_nights,maximum_nights,availability_30,availability_365,number_of_reviews,number_of_reviews_ltm,estimated_occupancy_l365d,estimated_revenue_l365d,review_scores_rating,review_scores_cleanliness,review_scores_location,instant_bookable,reviews_per_month
0,6422,2009-04-03,within a day,100%,NaN,f,1.0,['phone'],t,District 6,36.17143,-86.73570,Private room in home,Private room,1,1.0,1.0,1.0,"[""Dryer \u2013 In building"", ""Kayak"", ""Carbon ...",$43.00,30,365,0,175,667,0,0,0.0,4.95,4.96,4.92,f,3.34
1,39870,2010-07-18,within an hour,100%,91%,t,2.0,"['email', 'phone']",t,District 25,36.12466,-86.81269,Private room in home,Private room,2,1.0,1.0,1.0,"[""Carbon monoxide alarm"", ""Microwave"", ""Centra...",$70.00,1,7,17,242,587,80,255,17850.0,4.93,4.91,4.93,f,5.34


In [32]:
df['host_since'] = pd.to_datetime(df['host_since'], format='mixed')

In [33]:
df['host_response_rate'] = df['host_response_rate'].str.replace('%','').astype(float) / 100
df['host_acceptance_rate'] = df['host_acceptance_rate'].str.replace('%','').astype(float) / 100

In [34]:
#instead of injecting synthetic data its more preferred to drop since the missing values of this column account for approx 0.12
# price is the primary business metric and the target variable for analysis and price prediction. Imputing prices would
# introduce synthetic values and could bias the analysis.
df = df.dropna(subset=[
    'host_total_listings_count', 
    'host_identity_verified', 
    'host_since', 
    'price',
    'bathrooms',
    'beds'
]).astype({'host_total_listings_count': int})

In [35]:
df['price'] = df['price'].str.replace('$', '', regex=False).str.replace(',', '', regex=False).astype(float)

In [36]:
df['host_is_superhost'] = df['host_is_superhost'].map({'t': True, 'f': False}).astype('boolean')
df['host_identity_verified'] = df['host_identity_verified'].map({'t': True, 'f': False}).astype('boolean')
df['instant_bookable'] = df['instant_bookable'].map({'t': True, 'f': False}).astype('boolean')

In [37]:
df['bedrooms'] = df['bedrooms'].fillna(df.groupby('property_type')['bedrooms'].transform('median'))

In [38]:
review_cols = [
    'review_scores_rating',
    'review_scores_cleanliness',
    'review_scores_location',
    'reviews_per_month'
]

df[review_cols] = df[review_cols].fillna(0)
# No reviews exist. not saying the listing received a terrible rating. You're creating a sentinel value that clearly distinguishes "no reviews" from actual ratings

In [39]:
df['host_is_superhost'] = df['host_is_superhost'].fillna(False)

In [40]:
df['host_response_time'] = (df['host_response_time'].fillna('Unknown'))
# the instant bookable might be enabled or You genuinely do not know their response time so making a categorical value is the plausible way.

In [41]:
df['host_response_rate'] = (df['host_response_rate'].fillna(df['host_response_rate'].median()))
df['host_acceptance_rate'] = (df['host_acceptance_rate'].fillna(df['host_acceptance_rate'].median()))

In [42]:
# Check 
# df.info()
# (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
df.shape

(6629, 32)

In [43]:
df.to_csv('nashville_tennessee', index=False)

#### Step 4: feature engineering 

In [36]:
raw_df['estimated_revenue_l365d'].value_counts()

estimated_revenue_l365d
0.0        756
25500.0     14
23460.0     14
10800.0     14
20400.0     13
          ... 
648.0        1
2922.0       1
13896.0      1
702.0        1
1152.0       1
Name: count, Length: 2992, dtype: int64

In [41]:
# Shows counts for all values, including NaN
print(raw_df['estimated_revenue_l365d'].value_counts(dropna=False).head())


estimated_revenue_l365d
NaN        2809
0.0         756
10800.0      14
25500.0      14
23460.0      14
Name: count, dtype: int64


In [44]:
df.info()

<class 'pandas.DataFrame'>
Index: 6629 entries, 0 to 9442
Data columns (total 32 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   id                         6629 non-null   int64         
 1   host_since                 6629 non-null   datetime64[us]
 2   host_response_time         6629 non-null   str           
 3   host_response_rate         6629 non-null   float64       
 4   host_acceptance_rate       6629 non-null   float64       
 5   host_is_superhost          6629 non-null   boolean       
 6   host_total_listings_count  6629 non-null   int64         
 7   host_identity_verified     6629 non-null   boolean       
 8   neighbourhood_cleansed     6629 non-null   str           
 9   latitude                   6629 non-null   float64       
 10  longitude                  6629 non-null   float64       
 11  property_type              6629 non-null   str           
 12  room_type             